<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [ ]:
# TODO 0: 001에서 저장한 토크나이저를 로드하고 디바이스를 설정해봅시다.

import torch
import torch.nn as nn
from tokenizers import BertWordPieceTokenizer

tokenizer = BertWordPieceTokenizer("naver_wp-vocab.txt", lowercase=False, strip_accents=False)
vocab_size = tokenizer.get_vocab_size()
print(f"어휘 크기: {vocab_size:,}")

# 디바이스 자동 감지 (CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"디바이스: {device}")

</br>

# 학습 내용
>이번 장에서는 <strong>RNN (Recurrent Neural Network)</strong>을 활용하여 실제 한국어 감성 분류 문제를 학습합니다.</br></br>
>NSMC(네이버 영화 리뷰) 데이터셋으로 긍정/부정을 분류하는 모델을 직접 구현하고 학습해봅시다.</br></br>
>또한 <strong>Seq2Seq(Sequence-to-Sequence)</strong> 아키텍처의 인코더-디코더 구조도 이해해봅시다.

<div style="text-align:center">

</div>

</br>

# RNN (Recurrent Neural Network)
> RNN은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">시퀀스 데이터의 순서 정보를 기억</mark>하며 처리하는 신경망입니다.
> 이전 시점의 은닉 상태(Hidden State)를 현재 입력과 함께 사용합니다.

> 일반 신경망(MLP)은 고정 크기의 입력 벡터를 받아 고정 크기의 출력을 내므로, 입력 간의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">순서(시간적 의존성)</mark>를 고려하지 못합니다.</br></br>
> "나는 밥을 먹었다"와 "밥을 나는 먹었다"는 단어 집합은 같지만 의미가 다른데, MLP는 이 차이를 구분할 수 없어 언어, 음성, 시계열 데이터에 부적합합니다.</br></br>
> RNN의 핵심 아이디어는 은닉 상태(Hidden State)로 과거를 기억하는 것입니다.</br></br>
> RNN은 각 시점에서 현재 입력과 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이전 은닉 상태</mark>를 함께 받아 새로운 은닉 상태를 계산하며, 이 재귀적 구조 덕분에 시퀀스 앞부분의 정보가 뒷부분 처리에 영향을 미칩니다.</br></br>
> 임베딩(Embedding)을 통해 정수 인덱스를 밀집 벡터로 변환한 뒤, 역전파(Backpropagation)로 가중치를 조정하며 학습합니다.</br></br>
> 나아가 번역("나는 학생이다" → "I am a student")처럼 입출력 길이가 다른 문제에서는 단순 RNN을 직접 적용할 수 없습니다.</br></br>
> Seq2Seq는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인코더가 전체 입력을 하나의 컨텍스트 벡터로 압축</mark>하고, 디코더가 그 벡터로부터 출력을 순차 생성하는 방식으로 이 문제를 해결하며, 번역, 요약, 챗봇 등 현대 NLP 핵심 태스크의 기초 아키텍처입니다.

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b)$$

</br>

## nn.Embedding
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단어를 고정 크기의 밀집 벡터로 변환</mark>하는 룩업 테이블입니다.

In [ ]:
# TODO 1: 토크나이저의 vocab_size로 임베딩 레이어를 생성해봅시다.

In [ ]:
# TODO 2: 한국어 문장을 토크나이저로 인코딩해봅시다.

In [ ]:
# TODO 3: 인코딩된 정수 ID를 임베딩 벡터로 변환해봅시다.

</br>

## nn.RNN

<div style="text-align:center">

</div>

In [ ]:
# TODO 4: RNN 레이어를 생성하고, 임베딩 결과에 배치 차원을 추가하여 입력한 뒤 출력과 은닉 상태의 shape를 출력해봅시다.

💡nn.RNN 파라미터

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">파라미터</th>
      <th>설명</th>
      <th style="text-align:center">예시</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>input_size</code></td><td>입력 벡터의 차원 (임베딩 크기와 동일해야 함)</td><td style="text-align:center">256</td></tr>
    <tr><td style="text-align:center"><code>hidden_size</code></td><td>은닉 상태 벡터의 차원 (클수록 더 많은 정보 저장)</td><td style="text-align:center">512</td></tr>
    <tr><td style="text-align:center"><code>num_layers</code></td><td>RNN 층을 몇 개 쌓을지 (깊을수록 복잡한 패턴 학습)</td><td style="text-align:center">1</td></tr>
    <tr><td style="text-align:center"><code>batch_first</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">True</mark>면 입력 shape이 (배치, 시퀀스, 특성) 순서</td><td style="text-align:center">True</td></tr>
  </tbody>
</table>

> **출력 2가지:**</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">output</mark>: 모든 시점의 은닉 상태 → shape `(배치, 시퀀스길이, hidden_size)`</br>
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">hidden</mark>: 마지막 시점의 은닉 상태 → shape `(num_layers, 배치, hidden_size)` — 이것이 Seq2Seq의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">컨텍스트 벡터</mark>가 됩니다.

</br>

# NSMC 감성 분류 (RNN 분류기)
> NSMC(Naver Sentiment Movie Corpus) 데이터셋으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">긍정(1) / 부정(0)</mark>을 분류하는 RNN 모델을 학습합니다.</br></br>
> 입력 문장을 토크나이저로 정수 시퀀스로 변환하고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Embedding → RNN → FC</mark> 구조로 감성을 예측합니다.

In [ ]:
# TODO 5: ratings.txt를 로드하여 학습/테스트 데이터로 분리해봅시다.

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("ratings.txt", sep="\t")
df = df.dropna(subset=["document"])  # 결측값 제거
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f"전체 샘플 수: {len(df):,}")
print(f"학습: {len(train_df):,}  |  테스트: {len(test_df):,}")
print(f"\n샘플 확인:")
print(train_df[["document", "label"]].head(3).to_string(index=False))

💡패딩(Padding)이란?
> RNN은 배치 내 문장 길이가 모두 달라 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">길이를 통일해야 배치 처리</mark>가 가능합니다.</br></br>
> 짧은 문장은 `[PAD]` 토큰(ID=0)으로 뒤를 채우고, 긴 문장은 `max_length`로 자릅니다.</br></br>
> `collate_fn`은 DataLoader가 배치를 만들 때 호출되는 함수로, 가변 길이 시퀀스를 동일 길이로 맞춥니다.

In [ ]:
# TODO 6: NSMC 데이터셋 클래스를 정의해봅시다.

import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=64):
        # TODO 6-1: 텍스트 리스트와 레이블 리스트를 저장해봅시다.
        self.texts = df["document"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        # TODO 6-2: 데이터셋 크기를 반환해봅시다.
        return len(self.texts)

    def __getitem__(self, idx):
        # TODO 6-3: 텍스트를 토크나이저로 인코딩하고 max_length로 잘라봅시다.
        ids = self.tokenizer.encode(self.texts[idx]).ids[:self.max_length]
        label = self.labels[idx]
        return ids, label

In [ ]:
# TODO 7: collate_fn을 정의해봅시다.

def collate_fn(batch):
    # TODO 7-1: 배치 내 가장 긴 시퀀스 길이를 구해봅시다.
    ids_list, label_list = zip(*batch)
    max_len = max(len(ids) for ids in ids_list)  # 배치 내 최대 길이

    # TODO 7-2: 짧은 시퀀스는 0(PAD 토큰)으로 패딩해봅시다.
    padded = [ids + [0] * (max_len - len(ids)) for ids in ids_list]

    return (
        torch.LongTensor(padded),       # (batch, max_len)
        torch.LongTensor(label_list)    # (batch,)
    )

In [ ]:
# TODO 8: DataLoader를 만들어봅시다.

train_dataset = NSMCDataset(train_df, tokenizer)
test_dataset  = NSMCDataset(test_df,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, collate_fn=collate_fn)

print(f"학습 배치 수: {len(train_loader)}")
print(f"테스트 배치 수: {len(test_loader)}")

# 배치 shape 확인
sample_ids, sample_labels = next(iter(train_loader))
print(f"\n배치 shape — 입력: {sample_ids.shape}, 레이블: {sample_labels.shape}")

</br>

## RNN 감성 분류 모델
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Embedding → RNN → FC</mark> 구조로 문장의 감성을 예측합니다.</br></br>
> RNN의 마지막 은닉 상태(hidden state)가 문장 전체의 의미를 압축한 벡터로 사용됩니다.

In [ ]:
# TODO 9: RNN 감성 분류 모델 클래스를 정의해봅시다.

import torch.nn as nn

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # TODO 9-1: 임베딩 레이어를 초기화해봅시다. (padding_idx=0 → PAD 토큰은 학습 제외)
        # TODO 9-2: RNN 레이어를 초기화해봅시다.
        # TODO 9-3: 은닉 상태를 1개의 값으로 변환하는 선형 레이어를 초기화해봅시다.

    def forward(self, x):
        # TODO 9-4: 입력을 임베딩해봅시다.
        # TODO 9-5: RNN에 통과시켜 마지막 은닉 상태를 얻어봅시다.
        # TODO 9-6: 선형 레이어로 감성 로짓을 반환해봅시다.

In [ ]:
# TODO 10: 모델을 생성하고 샘플 배치로 테스트해봅시다.

💡BCEWithLogitsLoss란?
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">이진 교차 엔트로피(Binary Cross Entropy)</mark> 손실 함수입니다.</br></br>
> `sigmoid` 활성화와 `BCELoss`를 합친 수치적으로 안정적인 버전으로, 모델 출력에 sigmoid를 적용하지 않아도 됩니다.</br></br>
> 감성 분류처럼 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">출력이 1개인 이진 분류</mark>에 사용합니다.

In [ ]:
# TODO 11: 손실 함수와 옵티마이저를 정의해봅시다.

import torch.optim as optim

bce_loss  = nn.BCEWithLogitsLoss()             # 이진 분류용 손실 함수
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"손실 함수: {bce_loss}")
print(f"옵티마이저: {optimizer.__class__.__name__}")

</br>

## 학습 루프
> 에폭마다 전체 학습 데이터를 한 번씩 순회하며 모델의 가중치를 업데이트합니다.

In [ ]:
# TODO 12: 학습 루프를 구현해봅시다.

epochs = 5
train_loss_list     = []
train_accuracy_list = []

for epoch in range(epochs):
    model.train()
    running_loss  = 0.0
    correct_count = 0
    total_count   = 0

    for input_ids, labels in train_loader:
        input_ids = input_ids.to(device)
        labels    = labels.float().to(device)  # BCEWithLogitsLoss는 float 레이블 필요

        # TODO 12-1: 기울기를 초기화해봅시다.
        optimizer.zero_grad()                          # 이전 배치의 기울기 초기화
        # TODO 12-2: 순전파를 수행해봅시다.
        logits = model(input_ids)                      # 순전파: 입력 → 예측 로짓
        # TODO 12-3: 손실을 계산해봅시다.
        loss = bce_loss(logits, labels)                # 예측 vs 정답 비교 → 오차 측정
        # TODO 12-4: 역전파를 수행해봅시다.
        loss.backward()                                # 역전파: 각 파라미터의 기울기 계산
        # TODO 12-5: 파라미터를 업데이트해봅시다.
        optimizer.step()                               # 기울기 방향으로 파라미터 업데이트

        running_loss  += loss.item() * labels.size(0) # 배치 평균 × 샘플 수 = 배치 총 손실
        prediction     = (logits > 0).long()          # sigmoid > 0.5 와 동일
        correct_count += (prediction == labels.long()).sum().item()
        total_count   += labels.size(0)

    epoch_loss     = running_loss / total_count
    epoch_accuracy = correct_count / total_count
    train_loss_list.append(epoch_loss)
    train_accuracy_list.append(epoch_accuracy)
    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {epoch_loss:.4f}  Accuracy: {epoch_accuracy:.4f}")

In [ ]:
# TODO 13: 학습 손실과 정확도를 시각화해봅시다.

import matplotlib.pyplot as plt
import matplotlib

# 한글 폰트 설정
matplotlib.rcParams["font.family"] = "AppleGothic"
matplotlib.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, epochs + 1), train_loss_list, marker="o", color="#1565C0")
axes[0].set_title("학습 손실")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, epochs + 1), train_accuracy_list, marker="o", color="#2E7D32")
axes[1].set_title("학습 정확도")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

</br>

## 평가
> 학습된 모델을 테스트 데이터로 평가하고, 실제 리뷰 문장으로 감성 예측을 확인합니다.

In [ ]:
# TODO 14: 테스트 데이터로 모델 성능을 평가해봅시다.

model.eval()
correct_count = 0
total_count   = 0

with torch.no_grad():
    for input_ids, labels in test_loader:
        input_ids = input_ids.to(device)
        labels    = labels.to(device)

        # TODO 14-1: 순전파를 수행해봅시다.
        logits     = model(input_ids)         # 순전파
        # TODO 14-2: 예측값을 계산해봅시다. (logits > 0 이면 긍정)
        prediction = (logits > 0).long()      # 임계값 0 (sigmoid 0.5 와 동일)

        correct_count += (prediction == labels).sum().item()
        total_count   += labels.size(0)

test_accuracy = correct_count / total_count
print(f"테스트 정확도: {test_accuracy:.4f} ({correct_count}/{total_count})")

In [ ]:
# TODO 15: 실제 리뷰 문장으로 감성을 예측해봅시다.

def predict_sentiment(text):
    """문장을 입력받아 긍정/부정과 확률을 반환합니다."""
    model.eval()
    with torch.no_grad():
        ids = tokenizer.encode(text).ids[:64]
        input_tensor = torch.LongTensor([ids]).to(device)  # (1, seq_len)
        logit        = model(input_tensor)                  # (1,)
        probability  = torch.sigmoid(logit).item()          # 0~1 확률
    label = "긍정 😊" if probability >= 0.5 else "부정 😞"
    return label, probability

sample_reviews = [
    "정말 재미있고 감동적인 영화였습니다. 강력 추천합니다!",
    "돈과 시간이 아깝습니다. 최악의 영화입니다.",
    "배우들의 연기가 훌륭하고 스토리도 탄탄했어요.",
    "지루하고 스토리가 없음. 보다가 잠들었어요.",
]

print("=" * 60)
for review in sample_reviews:
    label, prob = predict_sentiment(review)
    print(f"[{label}] (확률: {prob:.3f})")
    print(f"  리뷰: {review[:50]}")
    print()

💡RNN의 한계
> 긴 시퀀스에서 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기울기 소실(Vanishing Gradient)</mark> 문제가 발생합니다.</br></br>
> 역전파 시 기울기가 시퀀스를 거슬러 올라가면서 점점 작아져, 초기 입력의 영향이 사라집니다.</br></br>
> 이를 해결하기 위해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LSTM(Long Short-Term Memory)</mark>과 GRU가 개발되었으며, 다음 노트북(003)에서 학습합니다.

<div style="text-align:center">

</div>

</br>

# Seq2Seq (Sequence-to-Sequence)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">인코더가 입력을 압축하고, 디코더가 출력을 생성</mark>하는 아키텍처입니다.

</br>

<div style="text-align:center">

</div>

## Encoder 구현

In [ ]:
# TODO 16: Encoder 클래스를 정의해봅시다.

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # TODO 16-1: 임베딩 레이어를 초기화해봅시다.
        # TODO 16-2: RNN 레이어를 초기화해봅시다.

    def forward(self, x):
        # TODO 16-3: 입력을 임베딩하고 RNN에 통과시켜 컨텍스트 벡터를 반환해봅시다.

In [ ]:
# TODO 17: Encoder 인스턴스를 생성하고, 실제 문장으로 테스트해봅시다.

</br>

## Decoder 구현

In [ ]:
# TODO 18: Decoder 클래스를 정의해봅시다.

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        # TODO 18-1: 임베딩 레이어를 초기화해봅시다.
        # TODO 18-2: RNN 레이어를 초기화해봅시다.
        # TODO 18-3: 은닉 상태를 vocab_size 크기로 변환하는 선형 레이어를 초기화해봅시다.

    def forward(self, x, hidden):
        # TODO 18-4: 입력을 임베딩해봅시다.
        # TODO 18-5: RNN에 임베딩과 이전 은닉 상태를 입력해봅시다.
        # TODO 18-6: 선형 레이어로 다음 토큰 확률을 예측해봅시다.

In [ ]:
# TODO 19: Decoder 인스턴스를 생성하고, 시작 토큰과 컨텍스트를 입력하여 예측 결과를 확인해봅시다.

</br>

## Seq2Seq 학습
> Encoder와 Decoder를 결합한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Seq2Seq</mark> 모델을 정의하고,
> 한국어 문장을 영어로 번역하는 태스크로 실제 학습을 진행합니다.</br></br>
> Decoder는 학습 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Teacher Forcing</mark>을 사용합니다 — 정답 토큰을 다음 입력으로 넣는 방식입니다.

In [ ]:
# TODO 20: Seq2Seq 클래스를 정의해봅시다.

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        # TODO 20-1: encoder와 decoder를 멤버 변수로 저장해봅시다.

    def forward(self, source, target):
        # TODO 20-2: Encoder에 source를 입력하여 context 벡터를 얻어봅시다.

        # TODO 20-3: Decoder에 target과 context를 입력하여 출력 로짓을 반환해봅시다.



In [ ]:
# TODO 21: 한영 번역 데이터를 정의해봅시다.

# 한국어→영어 번역 쌍 (15개)
pairs = [
    ("나는 학생입니다",      "I am a student"),
    ("오늘 날씨가 좋다",     "The weather is nice today"),
    ("이 영화 재미있다",     "This movie is fun"),
    ("고양이가 귀엽다",      "The cat is cute"),
    ("나는 밥을 먹는다",     "I eat rice"),
    ("그는 책을 읽는다",     "He reads a book"),
    ("우리는 학교에 간다",   "We go to school"),
    ("그녀는 노래를 부른다", "She sings a song"),
    ("나는 커피를 마신다",   "I drink coffee"),
    ("하늘이 파랗다",        "The sky is blue"),
    ("꽃이 아름답다",        "The flower is beautiful"),
    ("나는 음악을 듣는다",   "I listen to music"),
    ("비가 온다",            "It is raining"),
    ("그는 축구를 한다",     "He plays soccer"),
    ("나는 한국에 산다",     "I live in Korea"),
]

print(f"번역 쌍 {len(pairs)}개 정의 완료")
print(f"예시: '{pairs[0][0]}' → '{pairs[0][1]}'")

In [ ]:
# TODO 22: 소스/타겟 어휘집을 구축해봅시다.

# 소스 어휘집: NSMC 학습 토크나이저로 한국어 인코딩
# (TODO 0에서 로드한 tokenizer를 그대로 사용)
src_vocab_size = tokenizer.get_vocab_size()  # NSMC 토크나이저 vocab 크기

# 타겟 어휘집: 영어는 단어 단위 분리
# <PAD>=0: 패딩 (짧은 문장을 맞추기 위한 빈칸)
# <SOS>=1: Start Of Sequence (디코더 시작 신호)
# <EOS>=2: End Of Sequence (문장 끝 신호)
tgt_vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
for _, english in pairs:
    for word in english.split():
        if word not in tgt_vocab:
            tgt_vocab[word] = len(tgt_vocab)

tgt_idx_to_token = {idx: word for word, idx in tgt_vocab.items()}  # 역방향 매핑 (ID → 단어)
tgt_vocab_size = len(tgt_vocab)

print(f"소스 vocab 크기 (NSMC 토크나이저): {src_vocab_size}")
print(f"타겟 vocab 크기 (영어 단어 집합): {tgt_vocab_size}")
print(f"타겟 vocab 예시: {list(tgt_vocab.items())[:8]}")

In [ ]:
# TODO 23: Seq2Seq 모델을 생성하고 샘플 입력으로 테스트해봅시다.

embed_dim  = 64
hidden_dim = 128

seq2seq_encoder = Encoder(vocab_size=src_vocab_size, embed_dim=embed_dim, hidden_dim=hidden_dim)
seq2seq_decoder = Decoder(vocab_size=tgt_vocab_size, embed_dim=embed_dim, hidden_dim=hidden_dim)
seq2seq_model   = Seq2Seq(seq2seq_encoder, seq2seq_decoder).to(device)

# 샘플 입력으로 forward pass 테스트
sample_korean   = pairs[0][0]  # '나는 학생입니다'
sample_src_ids  = tokenizer.encode(sample_korean).ids
sample_source   = torch.LongTensor([sample_src_ids]).to(device)         # (1, src_len)
sample_tgt_ids  = [tgt_vocab['<SOS>']] + [tgt_vocab[w] for w in pairs[0][1].split()]
sample_target   = torch.LongTensor([sample_tgt_ids]).to(device)         # (1, tgt_len)

sample_output = seq2seq_model(sample_source, sample_target)
print(f"입력 문장   : {sample_korean}")
print(f"source shape: {sample_source.shape}")
print(f"target shape: {sample_target.shape}")
print(f"output shape: {sample_output.shape}  (batch=1, tgt_len, tgt_vocab_size={tgt_vocab_size})")


In [ ]:
# TODO 24: 손실 함수와 옵티마이저를 정의해봅시다.

import torch.optim as optim

# TODO 24-1: PAD 토큰(id=0)을 무시하는 CrossEntropyLoss를 cross_entropy에 저장해봅시다.
cross_entropy = nn.CrossEntropyLoss(ignore_index=tgt_vocab['<PAD>'])

# TODO 24-2: Adam 옵티마이저를 optimizer에 저장해봅시다. (lr=0.005)
optimizer = optim.Adam(seq2seq_model.parameters(), lr=0.005)

print(f"손실 함수: CrossEntropyLoss (ignore_index={tgt_vocab['<PAD>']}=<PAD>)")
print(f"옵티마이저: Adam (lr=0.005)")


In [ ]:
# TODO 25: Seq2Seq 학습 루프를 구현해봅시다.

epochs = 200
train_loss_list_seq2seq = []

for epoch in range(1, epochs + 1):
    seq2seq_model.train()
    running_loss = 0.0
    total_count  = 0

    for korean, english in pairs:
        # 소스: NSMC 토크나이저로 한국어 인코딩
        source_ids    = tokenizer.encode(korean).ids
        source_tensor = torch.LongTensor([source_ids]).to(device)  # (1, src_len)

        # 타겟: <SOS> + 영어 단어 + <EOS>
        # 예: "I am a student" → [<SOS>, I, am, a, student, <EOS>]
        target_ids    = [tgt_vocab['<SOS>']] + [tgt_vocab[w] for w in english.split()] + [tgt_vocab['<EOS>']]
        target_tensor = torch.LongTensor([target_ids]).to(device)  # (1, tgt_len)

        # ──── Teacher Forcing ────
        # 학습 시 Decoder에 정답 토큰을 다음 입력으로 제공하는 기법
        # 예: target = [<SOS>, I, am, a, student, <EOS>]
        #   decoder_input  = [<SOS>, I, am, a, student]  ← 마지막(<EOS>) 제외
        #   decoder_target = [I, am, a, student, <EOS>]  ← 첫 번째(<SOS>) 제외
        # Decoder는 <SOS>를 보고 "I"를 예측, "I"를 보고 "am"을 예측, ...
        decoder_input  = target_tensor[:, :-1]  # Decoder 입력: <SOS>부터 마지막 전까지
        decoder_target = target_tensor[:, 1:]   # 정답 레이블: 첫 토큰 다음부터 <EOS>까지

        # TODO 25-1: 기울기를 초기화해봅시다.
        optimizer.zero_grad()  # 이전 배치의 기울기 초기화

        # TODO 25-2: 순전파를 수행해봅시다.
        output = seq2seq_model(source_tensor, decoder_input)  # (1, tgt_len-1, tgt_vocab_size)

        # 손실 계산: CrossEntropyLoss는 (N, C) 형태 필요
        #   output shape:  (1, tgt_len-1, tgt_vocab_size)
        #   → squeeze(0) → (tgt_len-1, tgt_vocab_size)  ← 배치 차원 제거
        #   target shape:  (1, tgt_len-1)
        #   → squeeze(0) → (tgt_len-1,)                 ← 배치 차원 제거
        # TODO 25-3: 손실을 계산해봅시다.
        loss = cross_entropy(
            output.squeeze(0),         # (tgt_len-1, tgt_vocab_size)
            decoder_target.squeeze(0)  # (tgt_len-1,)
        )

        # TODO 25-4: 역전파를 수행해봅시다.
        loss.backward()   # 역전파: 각 파라미터의 기울기 계산

        # TODO 25-5: 파라미터를 업데이트해봅시다.
        optimizer.step()  # 기울기 방향으로 파라미터 업데이트

        token_count   = decoder_target.size(1)             # 이번 문장의 토큰 수
        running_loss += loss.item() * token_count          # 평균 손실 × 토큰 수 = 총 손실
        total_count  += token_count

    epoch_loss = running_loss / total_count                # 에폭 평균 손실
    train_loss_list_seq2seq.append(epoch_loss)

    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d}/{epochs}  loss: {epoch_loss:.4f}")

In [ ]:
# TODO 26: 학습 손실을 시각화해봅시다.

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(7, 3))
plt.plot(range(1, epochs + 1), train_loss_list_seq2seq)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Seq2Seq 학습 손실 (한영 번역)')
plt.tight_layout()
plt.show()

In [ ]:
# TODO 27: 번역 함수를 정의해봅시다.

# 학습 시에는 Teacher Forcing(정답 입력)을 사용했지만,
# 실제 번역 시에는 이전 예측 결과를 다음 입력으로 사용합니다. (Auto-regressive)

def translate(model, korean, src_tokenizer, tgt_vocab, tgt_idx_to_token, max_length=15):

    with torch.no_grad():
        # Encoder: 한국어 → 문맥 벡터


        # Decoder: <SOS>부터 시작하여 한 토큰씩 생성


        for _ in range(max_length):

            if predicted_token == '<EOS>':  # 문장 끝 신호가 나오면 종료


    return ' '.join(result_tokens)

In [ ]:
# TODO 28: 학습된 Seq2Seq 모델로 한국어 문장을 영어로 번역해봅시다.

# 학습 문장 번역 테스트
print("=== 학습 문장 번역 결과 ===")
for korean, english in pairs[:5]:
    result = translate(seq2seq_model, korean, tokenizer, tgt_vocab, tgt_idx_to_token)
    print(f"  한: {korean}")
    print(f"  영(정답): {english}")
    print(f"  영(예측): {result}")
    print()

# 새로운 문장 번역 테스트
print("=== 새로운 문장 번역 결과 ===")
new_sentences = ["나는 학생입니다", "하늘이 파랗다", "비가 온다"]
for korean in new_sentences:
    result = translate(seq2seq_model, korean, tokenizer, tgt_vocab, tgt_idx_to_token)
    print(f"  한: {korean} → 영: {result}")

💡Teacher Forcing이란?
> 학습 중 Decoder의 이전 스텝 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정답 토큰을 다음 입력으로 사용</mark>하는 기법입니다.</br></br>
> 모델이 예측에 실패해도 오류가 누적되지 않아 초반 학습 안정성이 높아집니다.</br></br>
> 실제 추론 시에는 Teacher Forcing 없이 이전 예측 토큰을 다음 입력으로 사용합니다.

💡왜 Seq2Seq가 필요한가?
> 입력과 출력의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">길이가 다른 경우</mark>(번역, 요약)에 고정 길이 모델로는 처리할 수 없습니다.</br>
> Seq2Seq는 가변 길이 입력을 가변 길이 출력으로 변환합니다.